
# ASSIGNMENT 11
# Fine-Tuning GPT-2 on a Custom Text Style

**Objective:** Understand how fine-tuning helps a pretrained GPT-2 language model adapt to a specific writing style.

### What this notebook does
1. Installs the required Hugging Face libraries.
2. Uploads your original custom text dataset.
3. Cleans the dataset by removing timestamps, usernames, links, and unnecessary formatting.
4. Checks the approximate number of lines.
5. Loads the small **GPT-2 (`gpt2`)** model.
6. Generates text using the original GPT-2 model.
7. Fine-tunes GPT-2 using the Hugging Face **Trainer**.
8. Generates text using the fine-tuned model.
9. Compares the original and fine-tuned outputs.
10. Saves the cleaned dataset, outputs, and fine-tuned checkpoint for submission.

### Recommended dataset
Use approximately **200–300 lines** of text with a consistent style:
- your own writing
- fictional conversations
- dialogues
- a suitable public-domain text
- another style-consistent text source

> **Important:** Use text you are permitted to use. Avoid uploading copyrighted song lyrics that you do not have rights to reproduce.


In [ ]:

# 1. Install required packages
!pip -q install -U transformers datasets accelerate

print("Required packages installed.")



## 2. Import libraries and check the runtime

For reasonable training speed, use **Runtime → Change runtime type → T4 GPU** in Google Colab when available.


In [ ]:

import os
import re
import json
import zipfile
import shutil
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. GPT-2 fine-tuning may be slow on CPU.")



## 3. Upload the original dataset

Create a `.txt` file containing approximately **200–300 lines** of consistent-style text.

Example:

```text
Hello, how are you today?
I am doing well. What about you?
The weather is beautiful outside.
...
```

Each line should contain useful language content rather than timestamps or usernames.


In [ ]:

print("Upload your original dataset (.txt):")
uploaded = files.upload()

txt_files = [name for name in uploaded.keys() if name.lower().endswith(".txt")]

if not txt_files:
    raise ValueError("Please upload a .txt dataset file.")

DATASET_FILE = txt_files[0]
print(f"\nUsing dataset: {DATASET_FILE}")


## 4. Read and inspect the original dataset

In [ ]:

with open(DATASET_FILE, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

raw_lines = [line.strip() for line in raw_text.splitlines()]

print("Original file size:", len(raw_text), "characters")
print("Original number of non-empty lines:", sum(bool(x) for x in raw_lines))

print("\nFirst 10 lines:")
for line in [x for x in raw_lines if x][:10]:
    print("•", line)



## 5. Clean the dataset

This cleaning step removes common non-language content such as:
- timestamps
- usernames
- URLs
- empty lines
- repeated whitespace
- simple metadata / formatting markers


In [ ]:

def clean_line(line):
    line = line.strip()

    if not line:
        return None

    # Common timestamp formats: [00:12], [01:23:45], 00:12, 01:23:45
    line = re.sub(r"\[?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?", "", line)

    # Common usernames / speaker metadata
    line = re.sub(r"@\w+", "", line)
    line = re.sub(r"^[A-Za-z0-9_.-]{1,30}\s*:\s*", "", line)

    # URLs
    line = re.sub(r"https?://\S+|www\.\S+", "", line)

    # Simple excessive formatting markers
    line = re.sub(r"^[\-*_=~#]+\s*", "", line)

    # Normalize whitespace
    line = re.sub(r"\s+", " ", line).strip()

    return line if line else None


cleaned_lines = []
for line in raw_lines:
    cleaned = clean_line(line)
    if cleaned:
        cleaned_lines.append(cleaned)

print("Cleaned number of lines:", len(cleaned_lines))

if len(cleaned_lines) < 150:
    print("WARNING: The cleaned dataset is quite small. The assignment recommends approximately 200–300 lines.")
elif len(cleaned_lines) > 350:
    print("NOTE: The dataset contains more than the recommended amount; this notebook can still train on it.")
else:
    print("Dataset size is in a reasonable range for this assignment.")



## 6. Save the cleaned dataset

This file will be used for fine-tuning and included in the submission package.


In [ ]:

CLEANED_DATASET = "cleaned_custom_style_dataset.txt"

with open(CLEANED_DATASET, "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print(f"Saved cleaned dataset as: {CLEANED_DATASET}")

print("\nFirst 15 cleaned lines:")
for line in cleaned_lines[:15]:
    print("•", line)


## 7. Load the pretrained GPT-2 model

In [ ]:

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 does not have a pad token by default.
tokenizer.pad_token = tokenizer.eos_token
base_model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token)



## 8. Enter the SAME prompt used in Assignment 2

Replace the example below with **exactly the same prompt** you previously used with the original GPT-2 model.

Keeping the prompt identical makes the before-and-after comparison fair.


In [ ]:

# ==========================================================
# IMPORTANT: REPLACE THIS WITH YOUR EXACT ASSIGNMENT 2 PROMPT
# ==========================================================

PROMPT = "The day started with"

print("Current prompt:")
print(PROMPT)


## 9. Generate text from the original GPT-2 model

In [ ]:

def generate_text(model, tokenizer_obj, prompt, max_new_tokens=120, temperature=0.8, top_p=0.95):
    model.eval()
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            pad_token_id=tokenizer_obj.eos_token_id
        )

    return tokenizer_obj.decode(output_ids[0], skip_special_tokens=True)


base_model.to(device)
base_output = generate_text(base_model, tokenizer, PROMPT)

print("===== ORIGINAL GPT-2 OUTPUT =====")
print(base_output)

with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)

print("\nSaved as base_model_output.txt")



## 10. Prepare the text dataset for GPT-2 fine-tuning

Each cleaned line is treated as one training example. The tokenizer converts text into GPT-2 tokens.


In [ ]:

dataset = Dataset.from_dict({"text": cleaned_lines})

print(dataset)
print(dataset[0])

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("\nTokenized dataset:")
print(tokenized_dataset)



## 11. Fine-tune GPT-2 using Hugging Face Trainer

The default settings are intentionally moderate for a Colab assignment.

**Suggested runtime:** GPU  
**Epochs:** 3  
**Batch size:** 4  
**Learning rate:** 5e-5


In [ ]:

OUTPUT_DIR = "./gpt2_finetuned_custom_style"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
train_result = trainer.train()

print("\nFine-tuning completed.")
print("Training metrics:")
print(train_result.metrics)


## 12. Save the fine-tuned model

In [ ]:

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Fine-tuned model saved to:", OUTPUT_DIR)



## 13. Generate text from the fine-tuned GPT-2 model

The **same prompt** from Assignment 2 is used again.


In [ ]:

fine_tuned_model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token
fine_tuned_model.config.pad_token_id = fine_tuned_tokenizer.pad_token_id
fine_tuned_model.to(device)

fine_tuned_output = generate_text(
    fine_tuned_model,
    fine_tuned_tokenizer,
    PROMPT
)

print("===== FINE-TUNED GPT-2 OUTPUT =====")
print(fine_tuned_output)

with open("fine_tuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(fine_tuned_output)

print("\nSaved as fine_tuned_model_output.txt")



## 14. Before-and-after comparison

### Original GPT-2
The original model has been trained on a broad text distribution, so its output may be more general and may not strongly reproduce the style of your custom dataset.

### Fine-tuned GPT-2
The fine-tuned model has been exposed to your custom text style, so its output should show stronger stylistic characteristics from the dataset.


In [ ]:

comparison = pd.DataFrame({
    "Aspect": [
        "Prompt Used",
        "Model",
        "Output File"
    ],
    "Original GPT-2": [
        PROMPT,
        MODEL_NAME,
        "base_model_output.txt"
    ],
    "Fine-Tuned GPT-2": [
        PROMPT,
        "GPT-2 fine-tuned on custom dataset",
        "fine_tuned_model_output.txt"
    ]
})

display(comparison)



## 15. Identify at least TWO clear style differences

Edit the two rows below using what you actually observe.

Suggested areas:
- vocabulary
- sentence length
- tone
- dialogue style
- repetition
- punctuation
- formatting
- emotional language
- use of phrases from the custom dataset


In [ ]:

# ==========================================================
# EDIT THESE TWO DIFFERENCES TO MATCH YOUR ACTUAL OUTPUTS
# ==========================================================

STYLE_DIFFERENCE_1 = (
    "The fine-tuned model uses vocabulary and phrases that are more similar "
    "to the writing style of the custom training dataset."
)

STYLE_DIFFERENCE_2 = (
    "The fine-tuned model follows the tone and sentence pattern of the custom dataset "
    "more closely, while the original GPT-2 remains more general."
)

analysis_table = pd.DataFrame({
    "Comparison Point": ["Style Difference 1", "Style Difference 2"],
    "Observation": [STYLE_DIFFERENCE_1, STYLE_DIFFERENCE_2]
})

display(analysis_table)



## 16. Optional simple text statistics

These statistics provide additional evidence for your discussion. They do not measure style quality directly.


In [ ]:

def text_stats(text):
    words = re.findall(r"\b\w+\b", text.lower())
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]

    return {
        "Characters": len(text),
        "Words": len(words),
        "Sentences": len(sentences),
        "Unique Words": len(set(words)),
        "Average Words/Sentence": round(len(words) / max(len(sentences), 1), 2)
    }

base_stats = text_stats(base_output)
fine_stats = text_stats(fine_tuned_output)

stats_df = pd.DataFrame({
    "Metric": list(base_stats.keys()),
    "Original GPT-2": list(base_stats.values()),
    "Fine-Tuned GPT-2": list(fine_stats.values())
})

display(stats_df)



# 17. Final Analysis / Conclusion

Edit the two observations above to match your actual results, then run the next cell. It creates an assignment-ready analysis.


In [ ]:

final_analysis = f"""
## Before-and-After Analysis

The original GPT-2 model produced a more general response because it was used without
fine-tuning on the custom writing style. After fine-tuning, the GPT-2 model adapted more
closely to the patterns present in the custom dataset.

The first clear difference was vocabulary and phrase usage. {STYLE_DIFFERENCE_1}

The second clear difference was tone and sentence structure. {STYLE_DIFFERENCE_2}

Overall, the experiment shows that fine-tuning can adapt a pretrained language model to a
specific writing style. The base GPT-2 provides general-purpose language generation, while
the fine-tuned version produces text that is more consistent with the style represented
in the custom dataset.
"""

print(final_analysis)

with open("before_after_analysis.txt", "w", encoding="utf-8") as f:
    f.write(final_analysis)



## 18. Create a submission package

This creates a ZIP containing:
- original dataset
- cleaned dataset
- base-model output
- fine-tuned model output
- before-and-after analysis
- comparison statistics

A second ZIP is created for the fine-tuned model checkpoint.


In [ ]:

# Save comparison statistics
comparison_table_file = "comparison_statistics.csv"
stats_df.to_csv(comparison_table_file, index=False)

# Package the textual submission files
submission_zip = "Assignment_11_Submission_Files.zip"

with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(DATASET_FILE, arcname=os.path.basename(DATASET_FILE))
    z.write(CLEANED_DATASET, arcname=os.path.basename(CLEANED_DATASET))
    z.write("base_model_output.txt", arcname="base_model_output.txt")
    z.write("fine_tuned_model_output.txt", arcname="fine_tuned_model_output.txt")
    z.write("before_after_analysis.txt", arcname="before_after_analysis.txt")
    z.write(comparison_table_file, arcname=comparison_table_file)

# Package the fine-tuned checkpoint
checkpoint_zip = "Assignment_11_Fine_Tuned_GPT2_Checkpoint.zip"

def zip_directory(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, dirs, files_list in os.walk(folder_path):
            for file_name in files_list:
                full_path = os.path.join(root, file_name)
                arcname = os.path.relpath(full_path, os.path.dirname(folder_path))
                z.write(full_path, arcname=arcname)

zip_directory(OUTPUT_DIR, checkpoint_zip)

print("Created:", submission_zip)
print("Created:", checkpoint_zip)



# Submission Checklist

Before submitting:

- [ ] Notebook `.ipynb`
- [ ] Original dataset (`.txt`) with approximately 200–300 lines
- [ ] Cleaned dataset
- [ ] Base-model output
- [ ] Fine-tuned model output
- [ ] Before-and-after comparison
- [ ] At least two clear style differences
- [ ] Fine-tuned model checkpoint, if required
- [ ] All notebook cells run successfully

### Important
Before submission:
1. Make sure **PROMPT** exactly matches the prompt from Assignment 2.
2. Replace the two example style differences with observations based on your actual outputs.
3. Save/download the notebook after running all cells so the outputs remain visible.
